# 🧠 HORec + MetaOpt: Higher-Order Recursive Kernel with Meta-Optimization

In [1]:
from __future__ import annotations
from typing import Callable, Any, List

State = Any
RecFun = Callable[[State], State]

class HORec:
    """
    Higher‑Order Recursive wrapper.
    • base  : the raw recursive function.
    • mods  : stack of meta‑filters that can rewrite outputs.
    • hist  : circular buffer of recent outputs for meta rules.
    """
    def __init__(self, base: RecFun, hist_len: int = 5):
        self.base, self.mods = base, []
        self.hist: List[State] = []
        self.H = hist_len

    def step(self, s: State) -> State:
        y = self.base(s)
        for f in self.mods:
            y = f(y, self.hist)
        self._upd_hist(y)
        return y

    def attach(self, f: Callable[[State, List[State]], State]):
        self.mods.append(f)

    def _upd_hist(self, y: State):
        self.hist.append(y)
        if len(self.hist) > self.H:
            self.hist.pop(0)

In [2]:
def anti_stagnate(output: State, hist: List[State]) -> State:
    if hist and output == hist[-1]:
        return output * 1.05
    return output

def momentum(output: State, hist: List[State], α: float = .7) -> State:
    if not hist: return output
    return α*output + (1-α)*hist[-1]

In [3]:
class MetaOpt:
    def __init__(self, core: HORec, λ: float = 2.9):
        self.core, self.λ = core, λ

    def __call__(self, x0: float, T: int = 40):
        x = x0
        for t in range(T):
            x = self.core.step(x)
            err = abs(x - .5)
            self.λ += 0.05 if err > 0.2 else -0.03
            self.λ = max(2.5, min(self.λ, 3.9))
            self.core.base = lambda z, l=self.λ: l*z*(1-z)
        return x

In [4]:
logistic = lambda z, λ=3.2: λ*z*(1-z)

core = HORec(logistic)
core.attach(anti_stagnate)
core.attach(momentum)

meta = MetaOpt(core, λ=3.2)
final_state = meta(.1)
print(final_state, core.hist[-5:])

0.6 [0.5999999999975477, 0.6000000000001225, 0.5999999999999939, 0.6000000000000003, 0.6]
